In [1]:
import os
import pandas as pd

In [ ]:
dataset_name='Beauty' 

os.chdir('./datasets/Amazon-2014/%s/5-core'%dataset_name)
os.getcwd()

In [3]:
# load item mapping
i_id_mapping = 'i_id_mapping.csv'
df = pd.read_csv(i_id_mapping, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (23033, 2)


,asin,itemID
0,0000031887,0
1,0123456479,1
2,1608299953,2
3,1617160377,3


In [4]:

import gzip, json
meta_file = '../raw/meta_%s.json.gz'%dataset_name

print('0 Extracting U-I interactions.')

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_file)

print(f'Total records: {meta_df.shape}')
meta_df[:3]

0 Extracting U-I interactions.
Total records: (1503384, 9)


,asin,related,title,price,salesRank,imUrl,brand,categories,description
0,0000037214,"{'also_viewed': ['B00JO8II76', 'B00DGN4R1Q', '...",Purple Sequin Tiny Dancer Tutu Ballet Dance Fa...,6.99,{'Clothing': 1233557},http://ecx.images-amazon.com/images/I/31mCncNu...,Big Dreams,"[[Clothing, Shoes & Jewelry, Girls], [Clothing...",NaN
1,0000031887,"{'also_bought': ['0000031852', '0000031895', '...",Ballet Dress-Up Fairy Tutu,6.79,{'Sports &amp; Outdoors': 8547},http://ecx.images-amazon.com/images/I/314qZjYe...,Boutique Cutie,"[[Clothing, Shoes & Jewelry, Girls, Clothing, ...",This adorable basic ballerina tutu is perfect ...
2,0123456479,"{'also_bought': ['B000BMTCK6', 'B0006JCGUM', '...",SHINING IMAGE HUGE PINK LEATHER JEWELRY BOX / ...,64.98,{'Kitchen & Dining': 16987},http://ecx.images-amazon.com/images/I/413tGhqo...,NaN,"[[Clothing, Shoes & Jewelry, Novelty, Costumes...",Elegance par excellence. Hand-crafted of the f...


In [ ]:
# remapping
map_dict = dict(zip(df['asin'], df['itemID']))

meta_df['itemID'] = meta_df['asin'].map(map_dict)
meta_df.dropna(subset=['itemID'], inplace=True)  
meta_df['itemID'] = meta_df['itemID'].astype('int64')
#meta_df['description'] = meta_df['description'].fillna(" ")
meta_df.sort_values(by=['itemID'], inplace=True)

print(f'shape: {meta_df.shape}')
meta_df[:2]

shape: (23033, 10)


,asin,related,title,price,salesRank,imUrl,brand,categories,description,itemID
1,0000031887,"{'also_bought': ['0000031852', '0000031895', '...",Ballet Dress-Up Fairy Tutu,6.79,{'Sports &amp; Outdoors': 8547},http://ecx.images-amazon.com/images/I/314qZjYe...,Boutique Cutie,"[[Clothing, Shoes & Jewelry, Girls, Clothing, ...",This adorable basic ballerina tutu is perfect ...,0
2,0123456479,"{'also_bought': ['B000BMTCK6', 'B0006JCGUM', '...",SHINING IMAGE HUGE PINK LEATHER JEWELRY BOX / ...,64.98,{'Kitchen & Dining': 16987},http://ecx.images-amazon.com/images/I/413tGhqo...,NaN,"[[Clothing, Shoes & Jewelry, Novelty, Costumes...",Elegance par excellence. Hand-crafted of the f...,1


In [6]:
ori_cols = meta_df.columns.tolist()

ret_cols = [ori_cols[-1]] + ori_cols[:-1]
print(f'new column names: {ret_cols}')
meta_df[:3]

new column names: ['itemID', 'asin', 'related', 'title', 'price', 'salesRank', 'imUrl', 'brand', 'categories', 'description']


,asin,related,title,price,salesRank,imUrl,brand,categories,description,itemID
1,0000031887,"{'also_bought': ['0000031852', '0000031895', '...",Ballet Dress-Up Fairy Tutu,6.79,{'Sports &amp; Outdoors': 8547},http://ecx.images-amazon.com/images/I/314qZjYe...,Boutique Cutie,"[[Clothing, Shoes & Jewelry, Girls, Clothing, ...",This adorable basic ballerina tutu is perfect ...,0
2,0123456479,"{'also_bought': ['B000BMTCK6', 'B0006JCGUM', '...",SHINING IMAGE HUGE PINK LEATHER JEWELRY BOX / ...,64.98,{'Kitchen & Dining': 16987},http://ecx.images-amazon.com/images/I/413tGhqo...,NaN,"[[Clothing, Shoes & Jewelry, Novelty, Costumes...",Elegance par excellence. Hand-crafted of the f...,1
36,1608299953,"{'also_bought': ['1572225289', '0877799172', '...",NaN,179.00,{'Software': 818},http://ecx.images-amazon.com/images/I/512RG9NT...,NaN,"[[Clothing, Shoes & Jewelry, Luggage & Travel ...",,2


In [7]:
ret_df = meta_df[ret_cols]
# dump
ret_df.to_csv(os.path.join('./', 'meta-%s.csv'%dataset_name), index=False)
print('done!')

done!


## Reload

In [8]:
indexed_df = pd.read_csv('meta-%s.csv'%dataset_name)
print(f'shape: {indexed_df.shape}')
indexed_df[:4]

shape: (23033, 10)


,itemID,asin,related,title,price,salesRank,imUrl,brand,categories,description
0,0,0000031887,"{'also_bought': ['0000031852', '0000031895', '...",Ballet Dress-Up Fairy Tutu,6.79,{'Sports &amp; Outdoors': 8547},http://ecx.images-amazon.com/images/I/314qZjYe...,Boutique Cutie,"[['Clothing, Shoes & Jewelry', 'Girls', 'Cloth...",This adorable basic ballerina tutu is perfect ...
1,1,0123456479,"{'also_bought': ['B000BMTCK6', 'B0006JCGUM', '...",SHINING IMAGE HUGE PINK LEATHER JEWELRY BOX / ...,64.98,{'Kitchen & Dining': 16987},http://ecx.images-amazon.com/images/I/413tGhqo...,NaN,"[['Clothing, Shoes & Jewelry', 'Novelty, Costu...",Elegance par excellence. Hand-crafted of the f...
2,2,1608299953,"{'also_bought': ['1572225289', '0877799172', '...",NaN,179.00,{'Software': 818},http://ecx.images-amazon.com/images/I/512RG9NT...,NaN,"[['Clothing, Shoes & Jewelry', 'Luggage & Trav...",NaN
3,3,1617160377,"{'also_bought': ['B002FPLE3U', '1423201787', '...",NaN,179.00,{'Software': 1137},http://ecx.images-amazon.com/images/I/51meMRCN...,NaN,"[['Clothing, Shoes & Jewelry', 'Luggage & Trav...",NaN


In [9]:
## Reload

i_uni = indexed_df['itemID'].unique()

print(f'# of unique items: {len(i_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(i_uni), max(i_uni)))

# of unique items: 23033
min/max of unique learners: 0/23032
